In [10]:
import cdsapi
import xarray as xr
import pandas as pd
import numpy as np
import zipfile
import os
import time
from pathlib import Path

OUTPUT_DIR = Path("./era5_test_3cities")
OUTPUT_DIR.mkdir(exist_ok=True)

CITIES = [
    {"id": "BO", "name": "Bologna", "lat": 44.50, "lon": 11.34},
    {"id": "TN", "name": "Trento",  "lat": 46.07, "lon": 11.12},
    {"id": "BA", "name": "Bari",    "lat": 41.12, "lon": 16.87},
]

START_YEAR = 1995
END_YEAR   = 2024
BBOX       = [47.0, 10.5, 40.5, 17.5]   # [North, West, South, East]

client = cdsapi.Client()
print("Connected ✓")

Connected ✓


In [11]:
def download_month(year: int, month: int) -> Path:
    zip_file    = OUTPUT_DIR / f"era5_raw_{year}_{month:02d}.zip"
    output_file = OUTPUT_DIR / f"era5_raw_{year}_{month:02d}.nc"

    if output_file.exists():
        print(f"  [SKIP] {year}-{month:02d} already extracted.")
        return output_file

    if not zip_file.exists():
        print(f"  [DOWNLOADING] {year}-{month:02d}...", end=" ", flush=True)
        client.retrieve(
            "reanalysis-era5-single-levels",
            {
                "product_type": "reanalysis",
                "variable": [
                    "2m_temperature",
                    "surface_solar_radiation_downwards",
                    "surface_net_solar_radiation_clear_sky",
                ],
                "year":  str(year),
                "month": f"{month:02d}",
                "day":   [f"{d:02d}" for d in range(1, 32)],
                "time":  [f"{h:02d}:00" for h in range(24)],
                "area":  BBOX,
                "format": "netcdf",
            },
            str(zip_file),
        )
        print("downloaded ✓", end=" ", flush=True)

    # Extract ALL .nc files from ZIP and merge into one dataset
    with zipfile.ZipFile(zip_file, "r") as z:
        names    = z.namelist()
        nc_names = [n for n in names if n.endswith(".nc")]
        print(f"\n  ZIP contains: {nc_names}")

        # Extract all files first
        for nc_name in nc_names:
            z.extract(nc_name, OUTPUT_DIR)

    # Open, merge, save — then explicitly close before deleting
    datasets   = []
    nc_paths   = [OUTPUT_DIR / nc_name for nc_name in nc_names]

    for nc_path in nc_paths:
        ds = xr.open_dataset(nc_path, engine="netcdf4")
        print(f"    {nc_path.name} → variables: {list(ds.data_vars)}")
        datasets.append(ds)

    ds_merged = xr.merge(datasets)
    ds_merged.to_netcdf(output_file)

    # Close ALL datasets before attempting to delete files
    for ds in datasets:
        ds.close()
    ds_merged.close()

    # Now safe to delete
    for nc_path in nc_paths:
        try:
            nc_path.unlink()
        except PermissionError:
            print(f"  [WARN] Could not delete {nc_path.name} — delete manually later")

    zip_file.unlink()
    print(f"  merged & saved ✓")
    return output_file

In [12]:
# CELL 3 — Extraction function (now with correct variable names)

def extract_cities(nc_file: Path) -> pd.DataFrame:
    ds       = xr.open_dataset(nc_file, engine="netcdf4")
    varnames = list(ds.data_vars)
    print(f"    Variables: {varnames}")

    # Flexible variable detection
    ssrd_var  = next((v for v in varnames if "ssrd" in v.lower()
                      and "cs" not in v.lower()), None)
    ssrdc_var = next((v for v in varnames if ("ssrd" in v.lower() and "cs" in v.lower())
                  or v.lower() in ("ssrdc", "ssrc", "fdir", "csr")), None)
    t2m_var   = next((v for v in varnames if "t2m" in v.lower()), None)

    # If still not found, print all and raise a clear error
    if not all([ssrd_var, ssrdc_var, t2m_var]):
        print(f"\n  Full dataset info:\n{ds}")
        raise ValueError(
            f"Could not map all variables.\n"
            f"  Found     : {varnames}\n"
            f"  Mapped to → T2m: {t2m_var} | SSRD: {ssrd_var} | SSRD_cs: {ssrdc_var}\n"
            f"  Check the variable names above and update the detection logic."
        )

    print(f"    Mapped → T2m: {t2m_var} | SSRD: {ssrd_var} | SSRD_cs: {ssrdc_var}")

    def deaccumulate(arr, times):
        arr = arr.astype(float)
        out = np.zeros_like(arr)
        for i in range(len(arr)):
            if times[i].hour == 1 or i == 0:
                out[i] = max(arr[i], 0) / 3600.0
            else:
                out[i] = max(arr[i] - arr[i - 1], 0) / 3600.0
        return out

    records = []

    for city in CITIES:
        ds_pt = ds.sel(latitude=city["lat"], longitude=city["lon"], method="nearest")

        time_coord = "valid_time" if "valid_time" in ds_pt.coords else "time"
        times      = pd.to_datetime(ds_pt[time_coord].values)

        ssrd_wm2  = deaccumulate(ds_pt[ssrd_var].values,  times)
        ssrdc_wm2 = deaccumulate(ds_pt[ssrdc_var].values, times)

        with np.errstate(divide="ignore", invalid="ignore"):
            ci = np.where(
                ssrdc_wm2 > 0,
                np.clip(ssrd_wm2 / ssrdc_wm2, 0, 1),
                0.0
            )

        records.append(pd.DataFrame({
            "time":        times,
            "city_id":     city["id"],
            "city":        city["name"],
            "lat":         city["lat"],
            "lon":         city["lon"],
            "T2m_C":       ds_pt[t2m_var].values - 273.15,
            "SSRD_Wm2":    ssrd_wm2,
            "SSRD_cs_Wm2": ssrdc_wm2,
            "CI":          ci,
        }))

    ds.close()
    return pd.concat(records, ignore_index=True)

In [ ]:
all_hourly = []

for year in range(START_YEAR, END_YEAR + 1):
    yearly = []

    for month in range(1, 13):
        print(f"\n[{year}-{month:02d}]")

        # Check if year-level parquet already exists (full year done)
        year_parquet = OUTPUT_DIR / f"hourly_3cities_{year}.parquet"
        if year_parquet.exists():
            print(f"  [SKIP] {year} parquet already saved.")
            yearly = None   # Signal to skip to next year
            break

        nc_file  = download_month(year, month)
        df_month = extract_cities(nc_file)
        yearly.append(df_month)

        # Delete raw NetCDF after extraction to save disk space
        nc_file.unlink()
        time.sleep(1)

    # Save year-level checkpoint parquet
    if yearly is not None and len(yearly) > 0:
        df_year = pd.concat(yearly, ignore_index=True)
        df_year.to_parquet(year_parquet, index=False)
        print(f"\n  ✓ {year} saved → {year_parquet.name} ({len(df_year):,} rows)")
        all_hourly.append(df_year)
    elif yearly is None:
        # Load from existing parquet
        all_hourly.append(pd.read_parquet(year_parquet))


In [17]:
df_all = pd.concat(all_hourly, ignore_index=True)
df_all = df_all.sort_values(["city_id", "time"]).reset_index(drop=True)

final_path = OUTPUT_DIR / "hourly_3cities_30y.parquet"
df_all.to_parquet(final_path, index=False)

print(f"\n{'='*50}")
print(f"✓ Final file  : {final_path}")
print(f"✓ Total rows  : {len(df_all):,}")
print(f"✓ Date range  : {df_all['time'].min()} → {df_all['time'].max()}")
print(f"✓ Cities      : {df_all['city'].unique().tolist()}")
print(f"✓ Columns     : {df_all.columns.tolist()}")
print(f"✓ File size   : {os.path.getsize(final_path) / 1024 / 1024:.1f} MB")
print(f"{'='*50}")

df_all.head(12)


✓ Final file  : era5_test_3cities\hourly_3cities_30y.parquet
✓ Total rows  : 788,976
✓ Date range  : 1995-01-01 00:00:00 → 2024-12-31 23:00:00
✓ Cities      : ['Bari', 'Bologna', 'Trento']
✓ Columns     : ['time', 'city_id', 'city', 'lat', 'lon', 'T2m_C', 'SSRD_Wm2', 'SSRD_cs_Wm2', 'CI']
✓ File size   : 10.0 MB


,time,city_id,city,lat,lon,T2m_C,SSRD_Wm2,SSRD_cs_Wm2,CI
0,1995-01-01 00:00:00,BA,Bari,41.12,16.87,14.185211,0.000000,0.000000,0.000000
1,1995-01-01 01:00:00,BA,Bari,41.12,16.87,13.901764,0.000000,0.000000,0.000000
2,1995-01-01 02:00:00,BA,Bari,41.12,16.87,13.980133,0.000000,0.000000,0.000000
3,1995-01-01 03:00:00,BA,Bari,41.12,16.87,14.487457,0.000000,0.000000,0.000000
4,1995-01-01 04:00:00,BA,Bari,41.12,16.87,14.536041,0.000000,0.000000,0.000000
5,1995-01-01 05:00:00,BA,Bari,41.12,16.87,14.964996,0.000000,0.000000,0.000000
6,1995-01-01 06:00:00,BA,Bari,41.12,16.87,15.064117,0.000000,0.000000,0.000000
7,1995-01-01 07:00:00,BA,Bari,41.12,16.87,14.908356,14.044444,12.977778,1.000000
8,1995-01-01 08:00:00,BA,Bari,41.12,16.87,15.132233,88.088889,91.804444,0.959527
9,1995-01-01 09:00:00,BA,Bari,41.12,16.87,15.701813,26.133333,113.475556,0.230299


In [34]:
df_all[(df_all['time'] > '2024-06-30') & (df_all['city'] == 'Bologna')].head(30)

,time,city_id,city,lat,lon,T2m_C,SSRD_Wm2,SSRD_cs_Wm2,CI,ssrd_condition
521545,2024-06-30 01:00:00,BO,Bologna,44.5,11.34,23.922272,0.000000,0.000000,0.0,Equal
521546,2024-06-30 02:00:00,BO,Bologna,44.5,11.34,23.313629,0.000000,0.000000,0.0,Equal
521547,2024-06-30 03:00:00,BO,Bologna,44.5,11.34,22.906158,0.000000,0.000000,0.0,Equal
521548,2024-06-30 04:00:00,BO,Bologna,44.5,11.34,22.568268,3.413333,2.666667,1.0,Cloud Enhancement
521549,2024-06-30 05:00:00,BO,Bologna,44.5,11.34,23.324615,72.071111,55.840000,1.0,Cloud Enhancement
521550,2024-06-30 06:00:00,BO,Bologna,44.5,11.34,25.154205,145.653333,119.928889,1.0,Cloud Enhancement
521551,2024-06-30 07:00:00,BO,Bologna,44.5,11.34,26.680084,162.595556,138.115556,1.0,Cloud Enhancement
521552,2024-06-30 08:00:00,BO,Bologna,44.5,11.34,27.632233,171.146667,139.288889,1.0,Cloud Enhancement
521553,2024-06-30 09:00:00,BO,Bologna,44.5,11.34,28.558990,146.275556,121.351111,1.0,Cloud Enhancement
521554,2024-06-30 10:00:00,BO,Bologna,44.5,11.34,29.297516,108.586667,89.813333,1.0,Cloud Enhancement


In [38]:
# Garantir formato datetime
df_all['time'] = pd.to_datetime(df_all['time'])

# Extrair apenas a hora
df_all['hour'] = df_all['time'].dt.hour

# Calcular percentual de zeros por hora
percentual_zeros = df_all.groupby('hour')['SSRD_Wm2'].apply(lambda x: (x == 0).mean() * 100)

print("Percentual de ocorrência de SSRD = 0 por hora (0-23h):")
print(percentual_zeros)

# Opcional: Remover coluna auxiliar
df_all.drop(columns=['hour'], inplace=True)

Percentual de ocorrência de SSRD = 0 por hora (0-23h):
hour
0     100.000000
1     100.000000
2     100.000000
3     100.000000
4      76.373426
5      51.454037
6      28.740038
7       3.336984
8       0.784815
9       2.181055
10      6.972075
11     18.649997
12     49.917868
13     82.825333
14     89.989049
15     94.585387
16     97.742897
17     99.379449
18     99.881365
19     99.993916
20    100.000000
21    100.000000
22     99.990874
23    100.000000
Name: SSRD_Wm2, dtype: float64


In [44]:
# Garantir formato datetime
df_all['time'] = pd.to_datetime(df_all['time'])

# Filtrar linhas onde a hora é 17 e a radiação é maior que 0
resultado = df_all[(df_all['time'].dt.hour == 18) & (df_all['SSRD_Wm2'] > 0)]

resultado

,time,city_id,city,lat,lon,T2m_C,SSRD_Wm2,SSRD_cs_Wm2,CI,ssrd_condition
35490,1999-01-18 18:00:00,BA,Bari,41.12,16.87,8.377106,0.000041,0.0,0.0,Cloud Enhancement
266850,1995-06-10 18:00:00,BO,Bologna,44.50,11.34,18.800934,28.711111,0.0,0.0,Cloud Enhancement
276306,1996-07-08 18:00:00,BO,Bologna,44.50,11.34,19.971588,20.906667,0.0,0.0,Cloud Enhancement
284274,1997-06-05 18:00:00,BO,Bologna,44.50,11.34,17.764801,5.404444,0.0,0.0,Cloud Enhancement
298482,1999-01-18 18:00:00,BO,Bologna,44.50,11.34,6.781403,0.000041,0.0,0.0,Cloud Enhancement
302154,1999-06-20 18:00:00,BO,Bologna,44.50,11.34,19.352448,15.306667,0.0,0.0,Cloud Enhancement
302610,1999-07-09 18:00:00,BO,Bologna,44.50,11.34,20.513580,9.102222,0.0,0.0,Cloud Enhancement
303234,1999-08-04 18:00:00,BO,Bologna,44.50,11.34,27.712311,31.626667,0.0,0.0,Cloud Enhancement
311058,2000-06-25 18:00:00,BO,Bologna,44.50,11.34,19.832672,17.511111,0.0,0.0,Cloud Enhancement
318570,2001-05-04 18:00:00,BO,Bologna,44.50,11.34,15.426666,3.111111,0.0,0.0,Cloud Enhancement


In [15]:
summary = (
    df_all
    .groupby("city")
    .agg(
        T2m_mean     = ("T2m_C",   "mean"),
        T2m_max      = ("T2m_C",   "max"),
        T2m_min      = ("T2m_C",   "min"),
        CI_mean      = ("CI",      "mean"),
        SSRD_mean_Wm2= ("SSRD_Wm2","mean"),
        n_hours      = ("T2m_C",   "count"),
    )
    .round(2)
)

print("\n30-Year Summary by City:")
print(summary.to_string())


30-Year Summary by City:
          T2m_mean    T2m_max  T2m_min  CI_mean  SSRD_mean_Wm2  n_hours
city                                                                   
Bari     16.440001  42.990002    -6.74     0.24          26.65   262992
Bologna  14.720000  40.220001   -13.57     0.24          24.27   262992
Trento    8.840000  30.690001   -23.25     0.23          23.22   262992


In [22]:
# Filter daytime only (both variables have radiation)
df_day = df_all[df_all['SSRD_cs_Wm2'] > 10]  # W/m² threshold, excludes twilight

df_day['ssrd_condition'] = df_day.apply(classify, axis=1)

summary_day = (df_day.groupby(['city', 'ssrd_condition'])
                     .size()
                     .unstack(fill_value=0))

summary_day['Total'] = summary_day.sum(axis=1)
for col in ['Cloud Enhancement', 'Normal', 'Equal']:
    if col in summary_day.columns:
        summary_day[f'{col} %'] = (summary_day[col] / summary_day['Total'] * 100).round(2)

print(summary_day)

ssrd_condition  Cloud Enhancement  Equal  Normal  Total  Cloud Enhancement %  \
city                                                                           
Bari                        42317     13   24259  66589                63.55   
Bologna                     42199     10   26645  68854                61.29   
Trento                      36911     12   31413  68336                54.01   

ssrd_condition  Normal %  Equal %  
city                               
Bari               36.43     0.02  
Bologna            38.70     0.01  
Trento             45.97     0.02  


C:\Users\LucasMonero\AppData\Local\Temp\ipykernel_30584\3046249372.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_day['ssrd_condition'] = df_day.apply(classify, axis=1)
